In [1]:
import cv2
import os

# 輸入影片檔
video_path = "test/2026_05_06_11_34_03_2_video.mp4"
output_dir = "test"
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

# 取得影片 FPS
fps = cap.get(cv2.CAP_PROP_FPS)
print("影片 FPS:", fps)

# 每秒要取的幀數
target_fps = 3

frame_interval = int(fps // target_fps)  # 每隔多少幀取一次
if frame_interval == 0:
    frame_interval = 1  # 避免 fps < target_fps 出錯

frame_count = 0
saved_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 按照間隔存檔
    if frame_count % frame_interval == 0:
        cv2.imwrite(f"{output_dir}/frame_{saved_count:05d}.png", frame)
        saved_count += 1
    
    frame_count += 1

cap.release()
print(f"總共擷取了 {saved_count} 幀")


影片 FPS: 20.0
總共擷取了 34 幀


In [ ]:
import cv2
import os
import numpy as np

# 輸入影片檔
video_path = "./2025_09_sonar_length_dataset/2025_09_10_16_35_00_2_video.mp4"
output_dir = "./ds3"
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("影片讀取失敗")
    exit()

frame_group = []
group_size = 3
saved_count = 0

# CLAHE 物件
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    # 去噪 (彩色影像可用 Gaussian)
    denoised = cv2.GaussianBlur(frame, (3,3), 0)

    frame_group.append(denoised.astype(np.float32))

    if len(frame_group) == group_size:
        # 平均重疊
        combined = np.mean(frame_group, axis=0).astype(np.uint8)

        # 對彩色影像做 CLAHE：轉 HSV 對 V 做增強
        hsv = cv2.cvtColor(combined, cv2.COLOR_BGR2HSV)
        hsv[:,:,2] = clahe.apply(hsv[:,:,2])
        enhanced = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # 存檔
        success = cv2.imwrite(f"{output_dir}/16_35_00_{saved_count:05d}.png", enhanced)
        if not success:
            print(f"寫入失敗: frame {saved_count}")

        saved_count += 1
        frame_group = []

cap.release()
print(f"總共擷取了 {saved_count} 張重疊增強彩色幀")

總共擷取了 66 張重疊增強彩色幀
